**The Lotka-Volterra predator-prey model.**
The Lotka-Volterra equations model the coupled population dynamics of
two interacting species - a prey population $x$ and a predator population $y$:

$$\frac{dx}{dt} = \alpha x - \beta x y
\qquad
\frac{dy}{dt} = \delta x y - \gamma y$$

Each term has a direct biological meaning:

- $\alpha x$ - prey reproduce at rate $\alpha$ when left alone
- $-\beta xy$ - prey are consumed by predators; rate proportional to encounters $xy$
- $\delta xy$ - predators gain population from eating prey
- $-\gamma y$ - predators die at rate $\gamma$ without food

The populations oscillate indefinitely with no friction or carrying capacity.
The **equilibrium point** (where both derivatives are zero) is found
by setting each equation to zero and solving.
From the prey equation:

$$0 = \alpha x - \beta x y = x(\alpha - \beta y)
\quad\Rightarrow\quad
y^* = \frac{\alpha}{\beta}$$

From the predator equation:

$$0 = \delta x y - \gamma y = y(\delta x - \gamma)
\quad\Rightarrow\quad
x^* = \frac{\gamma}{\delta}$$

(The trivial solution $x = y = 0$ - total extinction - is the other equilibrium.)
Both populations cycle around $(x^*, y^*)$ indefinitely.

This is a **coupled nonlinear ODE system** - the equations cannot be solved
independently because $dx/dt$ depends on $y$ and $dy/dt$ depends on $x$.
We use `scipy.integrate.solve_ivp` with the **RKF45** method
(adaptive Runge-Kutta 4th/5th order) which automatically adjusts
its step size to maintain accuracy.

---
**Model parameters and initial conditions.**
All four rate constants are positive.
Populations are expressed as fractions of a reference level
(1.0 = 100% of the reference population),
so the results will be plotted as percentages.

In [ ]:
"""predator_prey.ipynb"""

# Cell 01 - Model parameters and initial conditions

import inspect

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MultipleLocator
from scipy.integrate import solve_ivp

final_time = 20  # simulation duration (months)

alpha = 2.0  # prey birth rate (per month)
beta = 1.1  # prey death rate per predator-prey encounter
delta = 1.0  # predator birth rate per predator-prey encounter
gamma = 0.9  # predator death rate (per month)

pred_0 = 0.5  # initial predator population (50% of reference)
prey_0 = 1.0  # initial prey population    (100% of reference)

# Equilibrium point: populations cycle around (x*, y*)
prey_eq = gamma / delta  # x* = gamma/delta
pred_eq = alpha / beta  # y* = alpha/beta

print(f"\u03b1={alpha}, \u03b2={beta}, \u03b4={delta}, \u03b3={gamma}")
print(f"Initial predator : {pred_0:.2%}")
print(f"Initial prey     : {prey_0:.2%}")
print(f"Equilibrium prey : {prey_eq:.2%}  (x* = \u03b3/\u03b4)")
print(f"Equilibrium pred : {pred_eq:.2%}  (y* = \u03b1/\u03b2)")

**Defining the ODE system.**
`solve_ivp` expects a function `f(t, y)` that returns the derivatives
of the state vector.
The state vector here is `[pred, prey]`, so the function returns
`[d_pred/dt, d_prey/dt]` in the same order.
Extra constants ($\alpha, \beta, \delta, \gamma$) are passed through
the `args` keyword.

In [ ]:
# Cell 02 - Lotka-Volterra ODE system
# state_vector = [pred, prey]; returns [d_pred/dt, d_prey/dt]


def model(time, state_vector, alpha, beta, delta, gamma):
    pred, prey = state_vector
    d_prey = alpha * prey - beta * prey * pred  # dx/dt
    d_pred = delta * prey * pred - gamma * pred  # dy/dt
    return d_pred, d_prey


print(f"Function name  : {model.__name__}")
print(f"Signature      : {inspect.signature(model)}")

**Solving with `solve_ivp` (RKF45).**
The Runge-Kutta-Fehlberg method (RKF45) uses a 4th-order solution to
advance the state and a 5th-order solution to estimate the local error.
If the error exceeds a tolerance the step is rejected and retried with
a smaller $\Delta t$.
`max_step` caps the step size to ensure enough resolution even in
slow-varying regions.

In [ ]:
# Cell 03 - Solve the ODE system using RKF45

sol = solve_ivp(
    model,
    (0, final_time),  # time span
    [pred_0, prey_0],  # initial state vector [pred, prey]
    max_step=final_time / 1000,  # cap step size to 1/1000 of total time
    args=(alpha, beta, delta, gamma),
)

t = sol.t
pred, prey = sol.y * 100  # convert fractional populations to percent

pd.DataFrame({"time": t[:5], "predator %": pred[:5], "prey %": prey[:5]})

**Population dynamics over time.**
The predator and prey populations oscillate out of phase:
prey increase first (plentiful food, low predation),
which then drives predator growth (abundant prey),
which then suppresses the prey,
which then causes the predators to decline - and the cycle repeats.
The dashed horizontal lines mark the equilibrium values $x^*$ and $y^*$
around which both populations orbit.

In [ ]:
# Cell 04 - Plot predator and prey populations over time

plt.figure("predator_prey", figsize=(10, 5))
plt.plot(t, pred, label="Predator", color="red", linewidth=2)
plt.plot(t, prey, label="Prey", color="blue", linewidth=2)

# Mark the equilibrium levels
plt.axhline(
    prey_eq * 100,
    color="blue",
    linestyle=":",
    alpha=0.6,
    label=f"Prey equilibrium ({prey_eq:.0%})",
)
plt.axhline(
    pred_eq * 100,
    color="red",
    linestyle=":",
    alpha=0.6,
    label=f"Pred equilibrium ({pred_eq:.0%})",
)

plt.title("Predator-Prey Model (Lotka-Volterra)")
plt.xlabel("Time (months)")
plt.ylabel("Population (%)")

ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(MultipleLocator(1))
ax.yaxis.set_major_locator(MultipleLocator(50))
ax.yaxis.set_minor_locator(MultipleLocator(10))
ax.legend(loc="upper right", framealpha=1.0, facecolor="white")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()